# Apply Author Display-Name Curations

Detects which authors need a display_name re-sync and enqueues their work_ids. The curated display_name is applied as an **override** in `CreateAuthors` (via `COALESCE(curated, organic)`), not written in-place to `openalex.authors.authors`. This means deleting a curation automatically reverts the author to their organic name on the next `CreateAuthors` run.

Runs in `jobs/authors.yaml` after `SyncAuthorNameCurations` and before `CreateAuthors`.

```
SyncAuthorNameCurations
  ↓
ApplyAuthorNameCurations   ← this notebook (diff + enqueue)
  ↓
CreateAuthors              ← applies COALESCE(curated, organic) override
  ↓ openalex_authors
  ↓ next end2end cycle
UpdateWorkAuthorships re-emits works → sync_works → ES → API
```

## Diff logic

We compare each author's **currently displayed** name (`openalex_authors.display_name`) against what it **should be** (`COALESCE(curations.curated_display_name, authors.display_name)`). This single comparison catches three cases in one pass:
- **New curation**: displayed = organic, should = curated → differs → enqueue
- **Changed curation**: displayed = old curated, should = new curated → differs → enqueue
- **Deleted curation**: displayed = curated, should = organic (fallback) → differs → enqueue

## Stage diff (which authors' displayed name will change)

In [ ]:
%sql
-- Compare currently-displayed name vs what it should be under current curations.
-- Scoped to authors who either (a) have an active curation or (b) have a displayed
-- name that already differs from their organic name (i.e., a previously-applied or
-- now-deleted curation). The second condition catches curation deletions.
CREATE OR REPLACE TABLE openalex.authors.author_names_pending_changes AS
SELECT a.id AS author_id
FROM openalex.authors.authors a
JOIN openalex.authors.openalex_authors oa ON a.id = oa.id
LEFT JOIN openalex.authors.author_names_curations c ON a.id = c.author_id
WHERE (c.author_id IS NOT NULL OR NOT (oa.display_name <=> a.display_name))
  AND NOT (oa.display_name <=> COALESCE(c.curated_display_name, a.display_name))

## Enqueue affected work_ids for ES re-sync

In [ ]:
%sql
-- Enqueue every work_id where a display_name-changed author appears in work_authors.
INSERT INTO openalex.works.curated_work_ids_pending_sync
SELECT DISTINCT wa.work_id, current_timestamp() AS added_datetime
FROM openalex.works.work_authors wa
JOIN openalex.authors.author_names_pending_changes p ON wa.author_id = p.author_id

## Verify

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM openalex.authors.author_names_curations)       AS total_curations,
  (SELECT COUNT(*) FROM openalex.authors.author_names_pending_changes) AS authors_changed,
  (SELECT COUNT(DISTINCT wa.work_id)
     FROM openalex.works.work_authors wa
     JOIN openalex.authors.author_names_pending_changes p
       ON wa.author_id = p.author_id)                                  AS works_enqueued

In [ ]:
%sql
-- Spot-check: curated authors showing organic vs curated vs currently-displayed name.
SELECT
  c.author_id,
  a.display_name  AS organic_display_name,
  c.curated_display_name,
  oa.display_name AS currently_displayed
FROM openalex.authors.author_names_curations c
LEFT JOIN openalex.authors.authors a ON a.id = c.author_id
LEFT JOIN openalex.authors.openalex_authors oa ON oa.id = c.author_id
ORDER BY c.updated_datetime DESC NULLS LAST
LIMIT 100